In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ray

In [ ]:
# ============================================================
# SETTINGS
# ============================================================

N_REALIZATIONS = 10

noise_levels = [0.01, 0.03, 0.05, 0.10]

MODEL_PARAM_COUNT = 7

In [ ]:
# ============================================================
# TRUE MODEL
# ============================================================

true_parameters = [
    3.3e-02,
    8.2e-03,
    5.4e-02,
    1.07e+01,
    1.29e+01,
    1.21e+01,
    6.93e-01,
    1.05e+00,
    7.47e-01,
    2,
    3e-1,
    5,
    0.1,
    1,
    0.1,
    0.1,
]

In [ ]:
# ============================================================
# FORWARD MODEL
# ============================================================

objective_function = ParametersRecoveryRemote.remote(
    grid_params=grid_params
)

true_accretion_params = ParameterAccretionRate(
    mbh_par=2e11,
    alpha_par=3e-1,
    tau_par=5e9,
    eta=0.1,
    t_q=1e9,
    b_1=0.1,
    b_2=0.1,
)

outputs_true = ray.get(
    objective_function.directly_problem.remote(
        initial_condition,
        true_accretion_params
    )
)

true_final_data = outputs_true.n_final.copy()

In [ ]:
# ============================================================
# SINGLE REALIZATION
# ============================================================

def run_single_realization(noise_level, seed):

    # --------------------------------------------------------
    # noisy synthetic observations
    # --------------------------------------------------------

    target_data = add_noise_to_data(
        data=true_final_data,
        mu=0.0,
        sigma=noise_level,
        rho=0.5,
        alpha=0.5,
        seed=seed,
        noise_type="mixed"
    )

    # --------------------------------------------------------
    # inversion
    # --------------------------------------------------------

    result = ray.get(
        objective_function.optimizer.remote(
            bounds=bounds,
            target_data=target_data,
            fast_mode=False,
            weight_p=0.1,
            weight_theta=0.1,
        )
    )

    best_params = result.x

    # --------------------------------------------------------
    # recovered initial condition
    # --------------------------------------------------------

    gaussian_parameters = best_params[:-MODEL_PARAM_COUNT]

    recovered_initial = gaussian_sum_from_p(
        gaussian_parameters,
        ln_mbh_grid,
        n_gauss=3
    )

    # --------------------------------------------------------
    # reconstruction metrics
    # --------------------------------------------------------

    metrics = {
        "noise_level": noise_level,
        "seed": seed,

        "nrmse_initial": nrmse_percent(
            initial_condition,
            recovered_initial,
            denom="mean"
        ),

        "smape_initial": smape_percent(
            initial_condition,
            recovered_initial
        ),

        "mape_initial": stabilized_mape_percent(
            initial_condition,
            recovered_initial
        ),
    }

    return metrics

In [ ]:
# ============================================================
# MONTE CARLO LOOP
# ============================================================

all_results = []

for noise_level in noise_levels:

    print(f"\nNoise level = {noise_level:.2%}")

    for seed in range(N_REALIZATIONS):

        print(f"  realization {seed+1}/{N_REALIZATIONS}")

        metrics = run_single_realization(
            noise_level=noise_level,
            seed=seed
        )

        all_results.append(metrics)

In [ ]:
# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(all_results)

print(df.head())

In [ ]:
summary = (
    df.groupby("noise_level")
    .agg(
        {
            "nrmse_initial": ["mean", "std"],
            "smape_initial": ["mean", "std"],
            "mape_initial": ["mean", "std"],
        }
    )
)

print(summary)

In [ ]:
# ============================================================
# PLOT
# ============================================================

noise_percent = 100 * np.array(noise_levels)

mean_nrmse = (
    df.groupby("noise_level")["nrmse_initial"]
    .mean()
    .values
)

std_nrmse = (
    df.groupby("noise_level")["nrmse_initial"]
    .std()
    .values
)

plt.figure(figsize=(8, 6))

plt.plot(
    noise_percent,
    mean_nrmse,
    marker="o",
    linewidth=2,
    label="Mean NRMSE"
)

plt.fill_between(
    noise_percent,
    mean_nrmse - std_nrmse,
    mean_nrmse + std_nrmse,
    alpha=0.3,
    label=r"$\pm 1\sigma$"
)

plt.xlabel("Noise level (%)")
plt.ylabel("NRMSE (%)")

plt.title("Monte Carlo Noise Sensitivity Analysis")

plt.grid(True)
plt.legend()

plt.show()